# CE541E08 — Unit 4 · Day 31 — merge and join

| | |
|---|---|
| **Course** | CE541E08 |
| **Department** | Civil Engineering · Christ University |
| **Instructor** | Dr. Arpan Pradhan |
| **Unit** | Unit 4 — The Pandas Library |
| **Session** | Day 31 of 45 |
| **CO** | CO4 |
| **Topics** | pd.merge · inner/outer/left join · pd.concat · correlation |

---
> Read the explanation before each code block. Check the expected output. Run the cell and verify. Then try the small challenge at the end.
---

In [ ]:
student_name = "Your Full Name"
roll_number  = "2024XXXXXX"
session      = "Day 31"
print(f"CE541E08 | {student_name} | {roll_number} | {session}")

---
## Section 1 — Combining DataFrames

In hydrology, data comes from multiple sources — rainfall from IMD, streamflow from CWC, station metadata from a separate table. Pandas provides `pd.merge` (database-style join) and `pd.concat` (stacking) to combine them.

---
## Code Block 1 — pd.merge: Inner and Outer Joins

### What this code does

We merge a rainfall DataFrame and a streamflow DataFrame on a common Date column using inner and outer joins — showing how the number of rows changes with each join type.

### Why each step is taken

**`how='inner'`:**
Returns only dates that appear in **both** DataFrames. Dates with rainfall but no flow, or flow but no rainfall, are dropped. Use when you need complete pairs.

**`how='outer'`:**
Returns **all** dates from both DataFrames. Where one side has no data, the column is filled with `NaN`. Use when you want to see gaps.

**`how='left'`:**
Keeps all rows from the left (rainfall) DataFrame. Only adds flow where the date matches. Missing flow values become NaN. Useful when rainfall is your primary dataset and flow is supplementary.

### Algorithm

```
1. rainfall DataFrame: 5 dates, columns Date + Rainfall_mm
2. flow DataFrame: 5 dates (different set), columns Date + Flow_m3s

3. inner join on 'Date':
   → only dates in BOTH tables
   → drops rows unique to either table

4. outer join on 'Date':
   → all dates from both tables
   → NaN where one side is missing
```

### Expected output

```
Inner (both datasets):
         Date  Rainfall_mm  Flow_m3s
0  2024-06-01          0.0     234.5
1  2024-06-02         12.4     245.6
2  2024-06-03         45.6     312.4

Outer (all dates):
         Date  Rainfall_mm  Flow_m3s
0  2024-06-01          0.0     234.5
...
6  2024-06-06          NaN     567.8
```

In [ ]:
import pandas as pd

rainfall = pd.DataFrame({
    'Date':['2024-06-01','2024-06-02','2024-06-03','2024-06-04','2024-06-05'],
    'Rainfall_mm':[0.0, 12.4, 45.6, 0.0, 87.3],
})
flow = pd.DataFrame({
    'Date':['2024-06-01','2024-06-02','2024-06-03','2024-06-05','2024-06-06'],
    'Flow_m3s':[234.5, 245.6, 312.4, 456.7, 567.8],
})

# how='inner': keep only rows where Date appears in BOTH DataFrames
inner = pd.merge(rainfall, flow, on='Date', how='inner')

# how='outer': keep ALL dates from both; NaN where one side is missing
outer = pd.merge(rainfall, flow, on='Date', how='outer')

print("Inner (both datasets):"); print(inner)
print()
print("Outer (all dates):"); print(outer)

### 🔁 Try this

Try `how='left'` — this keeps all rainfall dates and only adds flow where available.

- How many rows does the left join return?
- Which dates have NaN in the Flow_m3s column?

---
## Code Block 2 — Merging Station Metadata with Peak Flow

### What this code does

We merge station metadata (name, state, catchment area) with a peak flow table using a Station_ID key column. After merging, we compute the unit peak discharge (peak per km² of catchment).

### Why each step is taken

**`how='left'`:**
We want to keep all peak flow records and add metadata where available. A left join on the peaks table ensures no peak records are lost.

**`combined['Peak_m3s'] / combined['CatchArea_km2']`:**
Computing a new column after merging. This gives the specific peak discharge — a useful normalised measure that allows comparison between catchments of different sizes.

### Algorithm

```
1. stations table: Station_ID, Name, State, CatchArea_km2
2. peaks table: Station_ID, Year, Peak_m3s (multiple years per station)

3. pd.merge(peaks, stations, on='Station_ID', how='left')
   → adds metadata to each peak record

4. combined['Unit_peak'] = Peak_m3s / CatchArea_km2
   → new column: peak per km²
```

### Expected output

```
           Station       State  Year  Peak_m3s  Unit_peak
0  Krishnarajasagara  Karnataka  2022      4567     0.6343
1     Cauvery Anicut  Tamil Nadu  2022     12340     0.1594
...
```

In [ ]:
import pandas as pd

stations = pd.DataFrame({
    'Station_ID'    : ['KRS','CANN','BILIG','METTUR'],
    'Station'       : ['Krishnarajasagara','Cauvery Anicut','Biligundlu','Mettur'],
    'State'         : ['Karnataka','Tamil Nadu','Karnataka','Tamil Nadu'],
    'CatchArea_km2' : [7200, 77450, 80000, 49800],
})
peaks = pd.DataFrame({
    'Station_ID': ['KRS','CANN','BILIG','METTUR','KRS','CANN'],
    'Year'      : [2022, 2022, 2022, 2022, 2023, 2023],
    'Peak_m3s'  : [4567, 12340, 15670, 8900, 3890, 10234],
})

# Left join: keep all peaks, add station metadata where ID matches
combined = pd.merge(peaks, stations, on='Station_ID', how='left')

# New column: unit peak discharge (m3/s per km2)
combined['Unit_peak'] = combined['Peak_m3s'] / combined['CatchArea_km2']

print(combined[['Station','State','Year','Peak_m3s','Unit_peak']].round(4))

### 🔁 Try this

Find the station with the highest unit peak in 2022 using:

`combined[combined['Year']==2022].nlargest(1,'Unit_peak')`

Which station has the most intense flood per unit area?

---
## Code Block 3 — pd.concat and Correlation

### What this code does

We use `pd.concat` to combine 3 years of monthly data into one DataFrame, then group by Year to get annual totals. We also compute the correlation between rainfall and streamflow.

### Why each step is taken

**`pd.concat([df1,df2,df3], ignore_index=True)`:**
Stacks DataFrames vertically (adds rows). `ignore_index=True` resets the row index to 0, 1, 2, ... instead of keeping each sub-DataFrame's original index. This is the right tool when the same columns appear in multiple DataFrames that need to be joined end-to-end.

**`df.groupby('Year')['Flow_m3s'].sum()`:**
After concatenating 3 years, `groupby('Year')` splits the data by year and `.sum()` computes the annual total for each year.

**`df.corr(numeric_only=True)`:**
Computes the Pearson correlation coefficient between all pairs of numeric columns. A value near +1 means strong positive correlation (more rain → more flow). `numeric_only=True` ignores text columns.

### Algorithm

```
1. Build 3 separate annual DataFrames (2022, 2023, 2024)

2. pd.concat([df1,df2,df3], ignore_index=True)
   → (36,3) DataFrame: 12 months × 3 years

3. .groupby('Year')['Flow_m3s'].sum()
   → annual total per year

4. df.corr(numeric_only=True)
   → correlation matrix between Flow_m3s and Rainfall_mm
```

### Expected output

```
Combined: (36, 3)
Annual totals:
Year
2022    ...
2023    ...
2024    ...

Correlation:
          Rainfall_mm  Flow_m3s
Rainfall_mm  1.000      0.987
Flow_m3s     0.987      1.000
```

In [ ]:
import pandas as pd, numpy as np

months = ['Jan','Feb','Mar','Apr','May','Jun',
          'Jul','Aug','Sep','Oct','Nov','Dec']
np.random.seed(42)
base  = [45,38,28,22,35,234,456,389,198,89,62,50]

# Build 3 separate annual DataFrames
dfs = []
for yr in [2022, 2023, 2024]:
    dfs.append(pd.DataFrame({
        'Year'    : yr,
        'Month'   : months,
        'Flow_m3s': np.round(np.array(base) + np.random.normal(0,30,12), 1)
    }))

# pd.concat stacks them vertically; ignore_index resets row numbering
all_yrs = pd.concat(dfs, ignore_index=True)
print(f"Combined: {all_yrs.shape}")
print(all_yrs.head(8))
print()
print("Annual totals:")
print(all_yrs.groupby('Year')['Flow_m3s'].sum().round(1))

In [ ]:
import pandas as pd, numpy as np

# Correlation between rainfall and streamflow
np.random.seed(10); n=60
df = pd.DataFrame({
    'Rainfall_mm': np.round(np.random.exponential(15, n), 1),
    'Flow_m3s'   : None,
}, index=pd.date_range('2024-06-01', periods=n, freq='D'))

df['Flow_m3s'] = np.round(df['Rainfall_mm']*12 + np.random.normal(100,30,n), 1)
print("Correlation:"); print(df.corr(numeric_only=True).round(3))

rainy = df[df['Rainfall_mm']>5].copy()
rainy['C_ratio'] = rainy['Flow_m3s'] / (rainy['Rainfall_mm']*12)
print(f"Mean runoff coeff: {rainy['C_ratio'].mean():.3f}")

### 🔁 Try this

In the correlation cell, add a third column `API3` (3-day antecedent rainfall) using:

`df['API3'] = df['Rainfall_mm'].rolling(3).sum().shift(1)`

Does the correlation between API3 and Flow_m3s improve?

---
## Session Summary — Merging and Combining

| Operation | Syntax | Result |
|---|---|---|
| Inner join | `pd.merge(a, b, on='key', how='inner')` | Only matching rows |
| Outer join | `pd.merge(a, b, on='key', how='outer')` | All rows, NaN for gaps |
| Left join | `pd.merge(a, b, on='key', how='left')` | All left rows |
| Stack rows | `pd.concat([a,b], ignore_index=True)` | Combine DataFrames vertically |
| Annual total | `df.groupby('Year')['col'].sum()` | Sum per year |
| Correlation | `df.corr(numeric_only=True)` | Pearson r matrix |
| New column | `df['new'] = df['a'] / df['b']` | Element-wise operation |

---
## Day 31 Assignment

10 days of rainfall and flow data — some dates don't match.

```python
rainfall = pd.DataFrame({'Date':['2024-07-01','2024-07-02','2024-07-03','2024-07-04',
                                  '2024-07-05','2024-07-06','2024-07-07','2024-07-08',
                                  '2024-07-09','2024-07-10'],
                         'Rainfall_mm':[0,12.4,45.6,0,87.3,134.5,22.3,0,67.8,45.1]})
flow = pd.DataFrame({'Date':['2024-07-01','2024-07-02','2024-07-03','2024-07-05',
                              '2024-07-06','2024-07-07','2024-07-09','2024-07-10'],
                     'Flow_m3s':[234,246,312,456,678,567,398,289]})
```

1. Perform inner, left, and outer merges — print the row count for each
2. On the inner merge result, compute the runoff ratio `C = Flow_m3s / Rainfall_mm` (skip zero-rainfall days)

### ▶ Assignment cell

In [ ]:
import pandas as pd

rainfall = pd.DataFrame({'Date':['2024-07-01','2024-07-02','2024-07-03','2024-07-04',
                                  '2024-07-05','2024-07-06','2024-07-07','2024-07-08',
                                  '2024-07-09','2024-07-10'],
                         'Rainfall_mm':[0,12.4,45.6,0,87.3,134.5,22.3,0,67.8,45.1]})
flow = pd.DataFrame({'Date':['2024-07-01','2024-07-02','2024-07-03','2024-07-05',
                              '2024-07-06','2024-07-07','2024-07-09','2024-07-10'],
                     'Flow_m3s':[234,246,312,456,678,567,398,289]})

inner = pd.merge(rainfall,flow,on='Date',how='inner')
left  = pd.merge(rainfall,flow,on='Date',how='left')
outer = pd.merge(rainfall,flow,on='Date',how='outer')
print(f"Inner:{len(inner)}  Left:{len(left)}  Outer:{len(outer)}")

inner['ratio'] = inner['Flow_m3s'] / inner['Rainfall_mm'].replace(0, float('nan'))
print(inner[['Date','Rainfall_mm','Flow_m3s','ratio']].round(2))

---
- [ ] Run all cells — verify outputs match expected outputs above
- [ ] Complete the assignment cell (replace `???` placeholders)
- [ ] Upload to GitHub: `Unit4_Pandas/CE541E08_U4_Day31.ipynb`
- [ ] Commit message: `Day 31 assignment completed`

*CE541E08 · Civil Engineering · Christ University · 2026-27 · Dr. Arpan Pradhan*